# 04 â€” Collaborative Filter

**Owner:** Arpan Chatterjee  
**Goal:** Build a user Ã— category collaborative filter using SVD  
to predict affinity scores for reward personalisation.

**Inputs:** `dataset/processed/receipts_master.csv`  
**Outputs:** `ml-service/models/collab_filter.pkl`  
**Target:** RMSE < 0.5 on held-out user-category pairs

In [1]:
# ── STEP 1: IMPORT LIBRARIES ──────────────────────────────────
# Surprise is a Python scikit for collaborative filtering (SVD, KNN, etc.)
import pandas as pd
import numpy as np
from surprise import SVD, Dataset, Reader, accuracy
from surprise.model_selection import cross_validate, train_test_split
import joblib

# Load synthetic user interactions (the only user-level dataset available)
df = pd.read_csv('../dataset/processed/synthetic_user_interactions.csv')

# Map category names to catalogue category strings ('grocery', 'food', 'retail')
import sys
sys.path.insert(0, '../ml-service')
import offers
df['category'] = df['category'].apply(offers.normalise_category)

print(f"Loaded {len(df)} interactions across {df['user_id'].nunique()} users.")
print(df.head())


Loaded 772 interactions across 60 users.
              user_id category                         merchant  amount  \
0  synthetic_user_001   retail  CROSS CHANNEL NETWORK SDN. BHD.    7.95   
1  synthetic_user_001   retail   UNIHAKKA INTERNATIONAL SDN BHD   12.20   
2  synthetic_user_001   retail                          UNKNOWN  134.00   
3  synthetic_user_001   retail                          UNKNOWN   38.00   
4  synthetic_user_001   retail                 AEON CO. (M) BHD   31.00   

   rating  is_synthetic  
0       3             1  
1       3             1  
2       5             1  
3       3             1  
4       5             1  


In [2]:
# ── STEP 2: BUILD SURPRISE DATASET ───────────────────────────
# Surprise expects a (user, item, rating) format
# Here: user=user_id, item=category, rating=rating
reader = Reader(rating_scale=(1, 5))
data = Dataset.load_from_df(df[['user_id', 'category', 'rating']], reader)
print("Surprise Dataset built successfully.")


Surprise Dataset built successfully.


In [3]:
# ── STEP 3: TRAIN SVD MODEL ───────────────────────────────────
# SVD (Singular Value Decomposition) factorises the user×category matrix
# into latent factors; 5-fold CV optimises RMSE and MAE
algo = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.02, random_state=42)
results = cross_validate(algo, data, measures=['RMSE', 'MAE'], cv=5, verbose=True)
print(f"Mean CV RMSE: {np.mean(results['test_rmse']):.4f}")


Evaluating RMSE, MAE of algorithm SVD on 5 split(s).

                  Fold 1  Fold 2  Fold 3  Fold 4  Fold 5  Mean    Std     
RMSE (testset)    0.8715  0.9207  0.9357  0.9672  0.8832  0.9157  0.0349  
MAE (testset)     0.7444  0.7788  0.7989  0.8316  0.7453  0.7798  0.0332  
Fit time          0.00    0.00    0.00    0.00    0.00    0.00    0.00    
Test time         0.00    0.00    0.00    0.00    0.00    0.00    0.00    
Mean CV RMSE: 0.9157


In [4]:
# ── STEP 4: TRAIN ON FULL DATASET & EXPORT ───────────────────
# Re-trains SVD on all available data (not just the CV split), then exports the
# fitted factors as plain arrays.
#
# WHY NOT joblib.dump(algo): pickling the Surprise estimator makes `surprise` a
# RUNTIME dependency of the ML service, because unpickling has to import the
# class. Surprise builds from Cython and is heavy to install on a cloud host, and
# if the import fails recommend.py silently falls back to content-based ranking —
# the model would look deployed while doing nothing. Exporting the arrays keeps
# serving dependency-free: prediction is arithmetic on numpy arrays.
trainset = data.build_full_trainset()
algo.fit(trainset)

import os
os.makedirs('../ml-service/models', exist_ok=True)

# Surprise's SVD prediction is:
#     est = global_mean + bu[u] + bi[i] + dot(qi[i], pu[u])
# with u, i the *inner* ids. Unknown users or items simply drop their term.
bundle = {
    "kind": "svd_factors",
    "global_mean": float(trainset.global_mean),
    "bu": algo.bu.astype("float32"),
    "bi": algo.bi.astype("float32"),
    "pu": algo.pu.astype("float32"),
    "qi": algo.qi.astype("float32"),
    "uid_map": {str(raw): int(trainset.to_inner_uid(raw))
                for raw in trainset._raw2inner_id_users},
    "iid_map": {str(raw): int(trainset.to_inner_iid(raw))
                for raw in trainset._raw2inner_id_items},
    "rating_scale": (1, 5),
    "n_factors": algo.n_factors,
    "trained_on": "synthetic_user_interactions.csv",
}
joblib.dump(bundle, '../ml-service/models/collab_filter.pkl')
print("Exported SVD factors to ../ml-service/models/collab_filter.pkl")
print(f"  users {len(bundle['uid_map'])} | items {len(bundle['iid_map'])} "
      f"| factors {bundle['n_factors']}")

# Verify the exported arithmetic reproduces Surprise's own prediction exactly,
# so the dependency-free path cannot drift from the trained model.
import numpy as np
def predict_from_bundle(b, uid, iid):
    est = b["global_mean"]
    u = b["uid_map"].get(str(uid))
    i = b["iid_map"].get(str(iid))
    if u is not None:
        est += b["bu"][u]
    if i is not None:
        est += b["bi"][i]
    if u is not None and i is not None:
        est += float(np.dot(b["qi"][i], b["pu"][u]))
    lo, hi = b["rating_scale"]
    return min(hi, max(lo, est))

worst = 0.0
for uid in list(bundle["uid_map"])[:20]:
    for iid in bundle["iid_map"]:
        worst = max(worst, abs(predict_from_bundle(bundle, uid, iid)
                               - algo.predict(uid, iid).est))
print(f"  max deviation from Surprise across 20 users x all items: {worst:.2e}")
assert worst < 1e-4, "exported factors do not reproduce Surprise predictions"


Exported SVD factors to ../ml-service/models/collab_filter.pkl
  users 60 | items 3 | factors 50
  max deviation from Surprise across 20 users x all items: 9.25e-08
